# Peters Dome, Alaska 2023 Remote Landslide

On 2023-09-14 an ice-rock avalanche of about 6.1 × 10⁶ m³ fell from Peters Dome in Denali National Park and Preserve, Alaska ([Toney et al., 2026, *JGR Earth Surface*](https://doi.org/10.1029/2025JF008577)). This notebook takes a first look at everything `gaia stage` writes for [peters-dome.yaml](peters-dome.yaml): the Copernicus 30 m DEM, Sentinel-1 radar and Sentinel-2 optical passes before and after the event, hourly ERA5 weather, and the seismic and infrasound records. The outlines of the source and of the source plus deposit ([peters-dome-sourcediff.geojson](peters-dome-sourcediff.geojson)) are from the USGS data release, [doi:10.5066/P13WIS6X](https://doi.org/10.5066/P13WIS6X).

## Stage

ERA5 needs `ARRAYLAKE_TOKEN` (a free Earthmover account). Staging everything takes a few minutes; with `--skip-existing`, running this again only stages what is missing.

In [ ]:
OUTPUT = "/tmp/gaia-scratch/gaia-cli/peters-dome"  # a local folder or an s3:// URL
!gaia stage --config peters-dome.yaml -o {OUTPUT} --skip-existing
# `!` ignores the exit code; IPython keeps it in _exit_code
assert _exit_code == 0, "gaia failed; see its log above"  # noqa: F821

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray  # noqa: F401  registers the .rio accessor
import xarray as xr
from scipy import signal

from gaia_cli import io, waveforms

STORE = f"{OUTPUT}/peters-dome.zarr"
EVENT = pd.Timestamp("2023-09-14T02:06:49")  # USGS ComCat origin time, UTC

tree = io.open_datatree(STORE)
outlines = gpd.read_file("peters-dome-sourcediff.geojson").set_index("name")  # source, total
print("\n".join(node.path for node in tree.subtree if node.data_vars))

## Terrain

The DEM (`dem/cop30`, longitude and latitude) over the 15 km `source` AOI, with the source outline and the source plus deposit (`total`) in magenta. The DEM predates the event: it is built from 2011–2015 TanDEM-X data.

In [ ]:
dem = tree["dem/cop30"].to_dataset()
fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
dem.elevation.plot.imshow(ax=ax, cmap="terrain", cbar_kwargs={"label": "elevation (m)"})
outlines.boundary.plot(ax=ax, color="magenta", lw=1)
ax.set_aspect(1 / np.cos(np.radians(63.14)))  # equal distances north and east
ax.set_title("Copernicus 30 m DEM");

## Sentinel-1: the change across the event

Each time step of `rasters/sentinel1-rtc` is one pass, with a `window` coordinate saying whether it is before (`pre`) or after (`post`) the event. The values are gamma0 in linear power. Here are the last pass before and the first after, on the same track, around the outlines (padded by 1.5 km), averaged over 2 × 2 cells (20 m) and compared in dB. The deposit, most of the total outline, brightens, and the source, at its southern end, darkens.

In [ ]:
s1 = tree["rasters/sentinel1-rtc"].to_dataset()
outlines_s1 = outlines.to_crs(s1.rio.crs)
west, south, east, north = outlines_s1.total_bounds
zoom = {"x": slice(west - 1500, east + 1500), "y": slice(north + 1500, south - 1500)}  # y decreases
pre = s1.vv.where(s1.window == "pre", drop=True).isel(time=-1)
post = s1.vv.where(s1.window == "post", drop=True).isel(time=0)
pre, post = (10 * np.log10(p.sel(**zoom).coarsen(x=2, y=2, boundary="trim").mean()) for p in (pre, post))

fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharex=True, sharey=True, layout="constrained")
for ax, scene, label in ((axes[0], pre, "before"), (axes[1], post, "after")):
    scene.plot.imshow(ax=ax, cmap="gray", vmin=-25, vmax=5, cbar_kwargs={"label": "γ⁰ VV (dB)"})
    ax.set_title(f"{label}: {pd.Timestamp(scene.time.values):%Y-%m-%d}")
(post - pre).plot.imshow(ax=axes[2], cmap="RdBu_r", vmin=-6, vmax=6, cbar_kwargs={"label": "after − before (dB)"})
axes[2].set_title("change")
for ax in axes:
    outlines_s1.boundary.plot(ax=ax, color="magenta", lw=1)
    ax.set_aspect("equal")
fig.suptitle(f"Sentinel-1 RTC VV at 20 m, relative orbit {int(s1.relative_orbit[0])}");

## Sentinel-2: before and after

`rasters/sentinel2` covers the 5 km `outlines` AOI at 10 m. Many passes are cloudy; these two, on the same relative orbit, are clear over the outlines: 2 September, before the event, and 22 September, after it, when the deposit is a dark streak on fresh snow. The bands are surface reflectance (0–1), stretched to 0–0.6 here.

The blue areas are snow and ice in the shadow of ridges. The passes are in the early afternoon, with the sun low in the south in September, so north-facing slopes, including the source, are shaded. In shadow, snow is lit only by the blue sky. The scene classification (`scl`) labels these areas as dark area or shadow (class 2).

In [ ]:
s2 = tree["rasters/sentinel2"].to_dataset()
fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharex=True, sharey=True, layout="constrained")
for ax, day in zip(axes, ["2023-09-02", "2023-09-22"]):
    scene = s2.sel(time=day).isel(time=0)
    rgb = xr.concat([scene.red, scene.green, scene.blue], dim="band") / 0.6
    rgb.clip(0, 1).plot.imshow(ax=ax, rgb="band")
    outlines.to_crs(s2.rio.crs).boundary.plot(ax=ax, color="magenta", lw=1)
    ax.set_title(f"{day} ({scene.window.item()})")
    ax.set_aspect("equal")
fig.suptitle("Sentinel-2 L2A, true colour");

## Weather from ERA5

Hourly ERA5 (`rasters/era5`) on its 0.25° grid, averaged over the cells around the outlines. The ERA5 simplified orography is about a kilometer below the actual mountainous elevation at the source location. As a result, temperatures there are overestimated (the source is colder), and the rainfall and snowfall distribution may be significantly off. `tp` (total precipitation) and `sf` (snowfall) are hourly totals in mm of water, so `tp − sf` is rain. The red line is the event.

In [ ]:
era5 = tree["rasters/era5"].to_dataset()[["t2m", "tp", "sf"]].mean(["x", "y"])
daily = era5.resample(time="1D").sum()

fig, (ax_t, ax_p) = plt.subplots(2, 1, figsize=(11, 6), sharex=True, layout="constrained")
era5.t2m.plot(ax=ax_t, color="tab:red", lw=0.8)
ax_t.axhline(0, color="k", lw=0.6)
ax_t.set(ylabel="2 m temperature (°C)", xlabel="", title="")
rain = (daily.tp - daily.sf).to_series()
snow = daily.sf.to_series()
ax_p.bar(rain.index, rain, width=0.8, align="edge", color="tab:green", label="rain")
ax_p.bar(snow.index, snow, width=0.8, align="edge", bottom=rain, color="tab:cyan", label="snowfall")
ax_p.set_ylabel("mm of water per day")
ax_p.legend(loc="upper right")
for ax in (ax_t, ax_p):
    ax.axvline(EVENT, color="red", lw=1.2)
fig.suptitle("ERA5 around Peters Dome");

## Seismic record section

`waveforms/seismic` holds raw counts at each channel's native rate, one group per rate, with `sites` and `channels` tables. `waveforms.to_grid` puts every channel on one 20 Hz grid in m/s, and `waveforms.by_component` arranges it by station and component. Here are the vertical components in two bands, as in the gaia-peters-dome report, each trace scaled to its own peak and placed at its distance from the source (`distance_km`, from the stage file's `origin`). The orange line is the ComCat origin time. In both bands the signal reaches farther stations later. At 15–100 s one wave train crosses the network at about 3 km/s, the speed of surface waves; at 1–5 Hz the energy lasts a minute or more at each station.

In [ ]:
def bandpass(da, low, high, rate=20):
    """*da* band-passed along time, its gaps filled with zeros."""
    sos = signal.butter(4, [low, high], btype="band", fs=rate, output="sos")
    filled = (da - da.mean("time")).fillna(0)
    return xr.apply_ufunc(signal.sosfiltfilt, sos, filled, input_core_dims=[[], ["time"]], output_core_dims=[["time"]])


def record_section(ax, traces, seconds, scale=8):
    """Each trace from *seconds* (start, end) after the event, scaled to its peak in
    that span and drawn at its distance."""
    traces = traces.sel(time=slice(*(EVENT + pd.Timedelta(seconds=s) for s in seconds)))
    t = (traces.time - np.datetime64(EVENT)) / np.timedelta64(1, "s")
    for trace in traces:
        ax.plot(t, trace.distance_km.item() + scale * trace / abs(trace).max(), color="k", lw=0.3)
    ax.axvline(0, color="tab:orange", lw=1.2)
    ax.set(xlabel="seconds after the ComCat origin", ylabel="distance (km)", xlim=seconds)


seismic = tree["waveforms/seismic"]
grid = waveforms.by_component(waveforms.to_grid(seismic, rate=20), seismic["sites"])
z = grid.velocity.sel(component="Z").sortby("distance_km")
fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=True, layout="constrained")
for ax, (low, high), band in ((axes[0], (1, 5), "1–5 Hz"), (axes[1], (1 / 100, 1 / 15), "15–100 s")):
    record_section(ax, bandpass(z, low, high), seconds=(-60, 240))
    ax.set_title(f"Vertical velocity, {band}")
axes[1].set_ylabel("")
fig.suptitle(f"{z.sizes['geometry']} seismic stations");

## Infrasound record section

`waveforms/infrasound` has the same layout, in pascals. Here is one channel per station, band-passed at 0.5–5 Hz. Each element of an IMS array is its own station, so I53US, near Fairbanks, shows as eight traces at about 252 km. The blue lines are arrival times for sound leaving the source at the ComCat origin and travelling at a constant celerity (horizontal distance / travel time) of 0.25 to 0.34 km/s. Most stations are noisy at these frequencies, but some show arrivals near the lines: the station at about 95 km near 300 s, and I53US.

In [ ]:
infrasound = tree["waveforms/infrasound"]
pressure = bandpass(waveforms.to_grid(infrasound, rate=20).pressure, 0.5, 5)
one_per_station = pressure.swap_dims(channel="station").drop_duplicates("station").sortby("distance_km")

fig, ax = plt.subplots(figsize=(11, 7), layout="constrained")
record_section(ax, one_per_station, seconds=(-300, 1800), scale=5)
d = np.array([0, 420])
for c, ls in ((0.34, "-"), (0.30, "--"), (0.25, ":")):
    ax.plot(d / c, d, color="tab:blue", lw=1, ls=ls, label=f"{c} km/s")
ax.set_ylim(0, 420)
ax.legend(title="celerity", loc="upper left")
ax.set_title(f"Pressure, 0.5–5 Hz, {one_per_station.sizes['station']} stations");

## The I53US array

The clearest arrival is at I53US. Its eight elements, each scaled to its own peak, show the main pulse about 870 s after the ComCat origin, an apparent celerity of about 0.29 km/s (solid line). Toney et al. (2026) report infrasound precursors in the minutes before the failure. ComCat lists one at 01:58:54 UTC, and shifting it by the same travel time (dashed line) lands on an earlier arrival near 400 s. A third arrival, near 710 s, has no ComCat entry; the [gaia-peters-dome](https://github.com/gaia-hazlab/gaia-peters-dome) report notes that this fits the two precursory events in the paper. This is a visual alignment only: the elements have not been combined (beamforming), which would also give the arrival's direction and speed across the array.

In [ ]:
PRECURSOR = pd.Timestamp("2023-09-14T01:58:54")  # ComCat precursor origin time, UTC
CELERITY = 0.29  # km/s

i53 = pressure.where(pressure.station.str.startswith("I53"), drop=True)
i53 = i53.sel(time=slice(EVENT + pd.Timedelta(seconds=300), EVENT + pd.Timedelta(seconds=1100)))
seconds = (i53.time - np.datetime64(EVENT)) / np.timedelta64(1, "s")
travel = float(i53.distance_km.mean()) / CELERITY

fig, ax = plt.subplots(figsize=(11, 5), layout="constrained")
for k, element in enumerate(i53):
    ax.plot(seconds, k + 0.45 * element / abs(element).max(), color="k", lw=0.4)
ax.axvline(travel, color="tab:blue", lw=1.2, label=f"ComCat origin + {travel:.0f} s")
ax.axvline((PRECURSOR - EVENT).total_seconds() + travel, color="tab:blue", lw=1.2, ls="--", label="ComCat precursor + the same")
ax.set_yticks(range(i53.sizes["channel"]), i53.station.values)
ax.set(xlabel="seconds after the ComCat origin", xlim=(300, 1100))
ax.set_ylim(-0.7, i53.sizes["channel"] + 0.5)
ax.legend(loc="upper center", ncols=2)
ax.set_title(f"IMS array I53US at {float(i53.distance_km.mean()):.0f} km, 0.5–5 Hz");